# 201 - Analytic Derivatives and Where They Pay Off

Every polynomial family includes `_der` and `_der_seq` functions.  Derivatives
are spatial (d/dx), not d/dn.  Several important "special" derivatives are
implemented, such as xy spatial derivatives of Zernike polynomials.  These are
needed for raytracing, but would also allow a modal shack hartmann wavefront
sensor to perform rapid reconstruction.

We'll show the derivative API, then compare against finite differences: analytic
derivatives are both more accurate and faster.

In [ ]:
import numpy as np
from matplotlib import pyplot as plt

from prysm.coordinates import make_xy_grid, cart_to_polar
from prysm.geometry import circle

# a 256x256 grid spanning the unit disk; r,t are polar; mask is True OUTSIDE
x, y = make_xy_grid(256, diameter=2)
r, t = cart_to_polar(x, y)
out = ~circle(1, r)        # boolean: True outside the unit circle (for blanking)

def show(arr, ax=None, blank=True, **kw):
    a = np.asarray(arr, dtype=float).copy()
    if blank:
        a[out] = np.nan
    ax = ax or plt.gca()
    return ax.imshow(a, cmap=kw.pop('cmap', 'RdBu'), **kw)

from prysm.polynomials import (
    zernike_nm_der, zernike_nm_der_xy, zernike_sum_der_xy,
    fringe_to_nm, Q2d_der_xy,
)

## Polar and Cartesian derivatives of a mode

`zernike_nm_der(n, m, r, t)` returns the radial and azimuthal derivatives
$(\partial_r, \partial_\theta)$; `zernike_nm_der_xy(n, m, x, y)` returns the
Cartesian $(\partial_x, \partial_y)$ - the form a raytrace wants, because surface
normals live in $x, y$.  Here are all four for coma.

In [ ]:
dr, dt = zernike_nm_der(3, 1, r, t)
dx, dy = zernike_nm_der_xy(3, 1, x, y)

fig, axs = plt.subplots(1, 4, figsize=(14, 3.4))
for ax, arr, lab in zip(axs, (dr, dt, dx, dy), ('d/dr', 'd/dt', 'd/dx', 'd/dy')):
    show(arr, ax); ax.set(title=f'coma {lab}', xticks=[], yticks=[])

## Sag and gradient in one pass

For a sum of modes, `zernike_sum_der_xy(coefs, nms, x, y)` returns the sag and
both Cartesian derivatives together, sharing the radial recurrence across all three:

In [ ]:
nms = [fringe_to_nm(j) for j in range(1, 37)]
rng = np.random.default_rng(2)
coefs = rng.standard_normal(36) / (1 + np.arange(36.0))**2
coefs[:3] = 0                                  # drop piston/tilt

sag, gx, gy = zernike_sum_der_xy(coefs, nms, x, y)
fig, axs = plt.subplots(1, 3, figsize=(12, 3.6))
for ax, arr, lab in zip(axs, (sag, gx, gy), ('sag', 'd/dx', 'd/dy')):
    show(arr, ax); ax.set(title=lab, xticks=[], yticks=[])

## Surface normals for a raytrace

The gradient is the surface normal.  For a sag $z(x,y)$ in implicit form, the
unit normal is $\hat{n} = (-z_x, -z_y, 1)/\sqrt{1 + z_x^2 + z_y^2}$.  With
analytic derivatives this is exact and cheap, essentially _de rigeur_ for
raytracing.  The normal can be visualized, as below.  The Z term looks quite
similar to the sag, because the mostly radial surface has sag mostly pointing
along z.

In [ ]:
norm = np.sqrt(1 + gx**2 + gy**2)
nx, ny, nz = -gx / norm, -gy / norm, 1.0 / norm

fig, axs = plt.subplots(1, 3, figsize=(12, 3.6))
for ax, arr, lab in zip(axs, (nx, ny, nz), ('n_x', 'n_y', 'n_z')):
    show(arr, ax); ax.set(title=f'surface normal {lab}', xticks=[], yticks=[])

## Analytic vs. finite difference

Finite differences can also be used, but require a second point to be
evaluated for each point.  There is also a potential numeric cancellation
error, and the step size has to be carefully chosen.  None of these issues
are present with analytical derivatives.

In [ ]:
h = x[0, 1] - x[0, 0]  # grid spacing
sag_only = zernike_sum_der_xy(coefs, nms, x, y)[0]
# central difference along x (axis 1)
fd_x = np.full_like(sag_only, np.nan)
fd_x[:, 1:-1] = (sag_only[:, 2:] - sag_only[:, :-2]) / (2 * h)

err = np.abs(fd_x - gx)
inside = circle(0.95, r)  # avoid the edge
print(f'max |analytic - finite-difference| inside pupil: {np.nanmax(err[inside]):.2e}')

fig, ax = plt.subplots(figsize=(4.5, 4))
e = err.copy(); e[out] = np.nan
im = ax.imshow(e, cmap='magma'); fig.colorbar(im, ax=ax, fraction=0.046)
ax.set(title='finite-difference truncation error', xticks=[], yticks=[])

The same `_der_xy` interface exists for the Forbes Q polynomials
(`Q2d_der_xy`) and every other family, so a freeform raytrace surface gets exact
normals the same way.  These are what the d/dx and d/dy of Q coma look like:

In [ ]:
qdx, qdy = Q2d_der_xy(1, 0, x, y)
fig, axs = plt.subplots(1, 2, figsize=(8, 3.6))
show(qdx, axs[0]); axs[0].set(title='Q2d(1,0) d/dx', xticks=[], yticks=[])
show(qdy, axs[1]); axs[1].set(title='Q2d(1,0) d/dy', xticks=[], yticks=[])

## Wrap-up

This completes the polynomials college.  The raytracing college's polynomial
surfaces and the interferometry college's Zernike fitting both build directly on
these functions